# HDB Resale Flat Prices ETL — Part 1

Part 1 of the HDB Senior Data Engineer technical test. ETL pipeline producing a cleaned master dataset of HDB resale flat transactions covering **January 2012 – December 2016**, sourced from [data.gov.sg collection 189](https://data.gov.sg/collections/189/view).

## How to use this notebook

Run the cells top to bottom. On a fresh clone the raw CSVs are already present in `data/raw/`, so the pipeline runs end-to-end without network access. The download path is still exercised when run, and is idempotent: files whose size matches the API's `datasetSize` are skipped.

All pipeline logic lives in `src/`. The notebook imports it and orchestrates each stage; heavy logic is intentionally kept out of cells.

## Stages mapped to the brief

| Brief requirement | Notebook section | Status |
|---|---|---|
| Dataset extraction (Jan 2012 – Dec 2016) | §1 Ingest | implemented |
| Combine into a single master dataset (DQ §1) | §2 Combine | implemented |
| Data profiling (DQ §2) | §3 Profile | pending |
| Validation rules on Date/Town/Flat Type/Flat Model/storey_range (DQ §3) | §4 Validate | pending |
| Recompute `remaining_lease` as of today (DQ §4) | §5 Clean | pending |
| Dedupe on composite key, keep highest price (DQ §5) | §5 Clean | pending |
| Anomaly heuristic on resale price (DQ §6) | §5 Clean | pending |
| Resale Identifier + hashing (Transformation §1–3) | — | **out of scope this pass** |

## 0. Setup

Configure logging and import the pipeline modules. Editable install (`pip install -e ".[dev]"`) makes `from src...` resolvable from the notebook working directory.

In [1]:
import logging

import pandas as pd

# INFO-level surfaces the per-stage progress lines emitted by src/ingest.py
# and src/combine.py directly into the notebook output.
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s: %(message)s",
    force=True,
)

from src import config
from src.ingest import ingest_all, verify_raw
from src.combine import combine_raw_files

print("project root:", config.PROJECT_ROOT)
print("scope window:", config.SCOPE_START, "..", config.SCOPE_END)
print("as_of_date:  ", config.AS_OF_DATE)

project root: /Users/ivanong/Documents/GitHub/hdb_resale_prices
scope window: 2012-01 .. 2016-12
as_of_date:   2026-04-08


## 1. Ingest

**Brief reference:** *Dataset Extraction* — *"Your ETL pipeline should process the data file as it is, without any manual modifications."*

**Goal:** Download the in-scope HDB resale CSVs from data.gov.sg into `data/raw/`.

**Approach:** Discover dataset IDs at runtime by walking the collection metadata endpoint and filtering on coverage window — no dataset IDs or filenames are hardcoded, in line with the brief's *"avoid hardcoding where possible"* directive. Downloads stream from the pre-signed S3 URLs returned by the `poll-download` endpoint, with 429 retry/backoff and `datasetSize`-based idempotency.

**Robustness:** If the API is unreachable, the pipeline falls back to whatever CSVs are already present in `data/raw/`. Because raw data is committed to the repo, the notebook is runnable on a fresh clone with no network access.

In [2]:
downloaded_paths = ingest_all()

print()
print(f"{len(downloaded_paths)} files in data/raw/:")
for p in downloaded_paths:
    print(f"  {p.name}  ({p.stat().st_size:,} bytes)")

2026-04-08 19:40:38,918 INFO src.ingest: Collection 189 has 5 child datasets
2026-04-08 19:40:40,605 INFO src.ingest: Filtered to 3 in-scope datasets for 2012-01..2016-12
2026-04-08 19:40:41,067 INFO src.ingest: poll-download rate-limited for d_43f493c6c50d54243cc1eab0df142d6a; sleeping 3.0s
2026-04-08 19:40:44,614 INFO src.ingest: Skip ResaleFlatPricesBasedonApprovalDate2000Feb2012.csv (already present, 29369945 bytes)
2026-04-08 19:40:47,165 INFO src.ingest: Skip ResaleFlatPricesBasedonRegistrationDateFromMar2012toDec2014.csv (already present, 4160771 bytes)
2026-04-08 19:40:49,389 INFO src.ingest: poll-download rate-limited for d_ea9ed51da2787afaf8e51f827c304208; sleeping 3.0s
2026-04-08 19:40:52,632 INFO src.ingest: poll-download rate-limited for d_ea9ed51da2787afaf8e51f827c304208; sleeping 3.0s
2026-04-08 19:40:56,231 INFO src.ingest: Skip ResaleFlatPricesBasedonRegistrationDateFromJan2015toDec2016.csv (already present, 3070924 bytes)



3 files in data/raw/:
  ResaleFlatPricesBasedonApprovalDate2000Feb2012.csv  (29,369,945 bytes)
  ResaleFlatPricesBasedonRegistrationDateFromMar2012toDec2014.csv  (4,160,771 bytes)
  ResaleFlatPricesBasedonRegistrationDateFromJan2015toDec2016.csv  (3,070,924 bytes)


### 1.1 Verify

For each CSV in `data/raw/`, parse it with pandas and check three things against the API's authoritative metadata:

| Check | What it confirms |
|---|---|
| **size_ok** | File size equals the API's `datasetSize` exactly |
| **cols_ok** | Column names equal `columnMetadata` (in order) |
| **dates_ok** | Every row's `month` falls inside the dataset's coverage window |

Together these form the strongest integrity envelope available without a server-published checksum. `verify_raw()` also reports any in-scope dataset that has no local file, so the inverse failure mode is caught too.

In [3]:
verification = verify_raw()
verification_df = pd.DataFrame(verification)

display(verification_df[[
    "file", "matched_dataset", "rows", "cols",
    "min_month", "max_month",
    "actual_size", "expected_size",
    "size_ok", "cols_ok", "dates_ok", "ok",
]])

assert all(r["ok"] for r in verification), "Stage 1 verification failed — see logs above"

2026-04-08 19:40:56,471 INFO src.ingest: Collection 189 has 5 child datasets
2026-04-08 19:40:58,069 INFO src.ingest: Filtered to 3 in-scope datasets for 2012-01..2016-12
2026-04-08 19:40:58,451 INFO src.ingest: [OK] ResaleFlatPricesBasedonApprovalDate2000Feb2012.csv  rows=369651 cols=10  2000-01..2012-02  size_ok=True cols_ok=True dates_ok=True
2026-04-08 19:40:58,490 INFO src.ingest: [OK] ResaleFlatPricesBasedonRegistrationDateFromJan2015toDec2016.csv  rows=37153 cols=11  2015-01..2016-12  size_ok=True cols_ok=True dates_ok=True
2026-04-08 19:40:58,541 INFO src.ingest: [OK] ResaleFlatPricesBasedonRegistrationDateFromMar2012toDec2014.csv  rows=52203 cols=10  2012-03..2014-12  size_ok=True cols_ok=True dates_ok=True
2026-04-08 19:40:58,542 INFO src.ingest: verify_raw: 3 files checked, overall OK


,file,matched_dataset,rows,cols,min_month,max_month,actual_size,expected_size,size_ok,cols_ok,dates_ok,ok
0,ResaleFlatPricesBasedonApprovalDate2000Feb2012...,d_43f493c6c50d54243cc1eab0df142d6a,369651,10,2000-01,2012-02,29369945,29369945,True,True,True,True
1,ResaleFlatPricesBasedonRegistrationDateFromJan...,d_ea9ed51da2787afaf8e51f827c304208,37153,11,2015-01,2016-12,3070924,3070924,True,True,True,True
2,ResaleFlatPricesBasedonRegistrationDateFromMar...,d_2d5ff9ea31397b66239f245f57751537,52203,10,2012-03,2014-12,4160771,4160771,True,True,True,True


## 2. Combine

**Brief reference:** *Data Quality Requirements §1* — *"Combine the datasets into a single master dataset. The combined dataset should contain all the attributes found in all files."*

**Goal:** Union the verified raw CSVs into a single master DataFrame ready for profiling, validation, and cleaning.

**Per-file pipeline:**

1. **Normalize headers** — `strip().lower().replace(" ", "_")` on every column on read. Defends against case/whitespace drift across vintages so the schema union doesn't produce duplicates like `Month` vs `month`.
2. **Filter to scope** — drop rows whose `month` falls outside `SCOPE_START..SCOPE_END`. For the 2000–Feb 2012 file this trims to its Jan–Feb 2012 tail; for the other two it's a no-op. Applying it uniformly avoids special-casing.
3. **Normalize `remaining_lease`** — the Jan 2015–Dec 2016 file stores it as integer years already (verified by inspection: dtype `int64`, sample values like 70, 65, 64). We copy it into a canonical `remaining_lease_years_original` `Int64` column. Pre-2015 vintages don't have this column at all, so the master will hold `NaN` for those rows. The 2017+ "X years Y months" string format is out of scope and not parsed here.
4. **Tag with `source_file`** — lineage column for downstream debugging.

Frames are then concatenated with `sort=False`, taking the union of columns (pandas fills `NaN` where a column is absent in any particular frame).

In [4]:
master = combine_raw_files()

print()
print(f"Master shape: {master.shape[0]:,} rows x {master.shape[1]} columns")
print()
print("Rows per source file:")
print(master.groupby("source_file").size().to_string())
print()
print(f"Month range: {master['month'].min()} .. {master['month'].max()}")
print()
print("remaining_lease_years_original by source file:")
print(
    master.groupby("source_file")["remaining_lease_years_original"]
    .apply(lambda s: f"{s.notna().sum():>6,d} / {len(s):>6,d} populated")
    .to_string()
)

2026-04-08 19:41:03,923 INFO src.combine: ResaleFlatPricesBasedonApprovalDate2000Feb2012.csv: 369651 rows raw, 3188 in scope (2012-01..2016-12)
2026-04-08 19:41:03,964 INFO src.combine: ResaleFlatPricesBasedonRegistrationDateFromJan2015toDec2016.csv: 37153 rows raw, 37153 in scope (2012-01..2016-12)
2026-04-08 19:41:04,019 INFO src.combine: ResaleFlatPricesBasedonRegistrationDateFromMar2012toDec2014.csv: 52203 rows raw, 52203 in scope (2012-01..2016-12)
2026-04-08 19:41:04,025 INFO src.combine: Combined master: 92544 rows, 13 columns from 3 files



Master shape: 92,544 rows x 13 columns

Rows per source file:
source_file
ResaleFlatPricesBasedonApprovalDate2000Feb2012.csv                  3188
ResaleFlatPricesBasedonRegistrationDateFromJan2015toDec2016.csv    37153
ResaleFlatPricesBasedonRegistrationDateFromMar2012toDec2014.csv    52203

Month range: 2012-01 .. 2016-12

remaining_lease_years_original by source file:
source_file
ResaleFlatPricesBasedonApprovalDate2000Feb2012.csv                      0 /  3,188 populated
ResaleFlatPricesBasedonRegistrationDateFromJan2015toDec2016.csv    37,153 / 37,153 populated
ResaleFlatPricesBasedonRegistrationDateFromMar2012toDec2014.csv         0 / 52,203 populated


### 2.1 Sample

First five rows of the combined master, with all 13 columns:

In [5]:
display(master.head())

,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price,source_file,remaining_lease,remaining_lease_years_original
0,2012-01,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,01 TO 03,44.0,Improved,1979,257800.0,ResaleFlatPricesBasedonApprovalDate2000Feb2012...,NaN,<NA>
1,2012-01,ANG MO KIO,2 ROOM,314,ANG MO KIO AVE 3,07 TO 09,44.0,Improved,1978,263000.0,ResaleFlatPricesBasedonApprovalDate2000Feb2012...,NaN,<NA>
2,2012-01,ANG MO KIO,2 ROOM,314,ANG MO KIO AVE 3,10 TO 12,44.0,Improved,1978,275000.0,ResaleFlatPricesBasedonApprovalDate2000Feb2012...,NaN,<NA>
3,2012-01,ANG MO KIO,2 ROOM,170,ANG MO KIO AVE 4,01 TO 03,45.0,Improved,1986,260000.0,ResaleFlatPricesBasedonApprovalDate2000Feb2012...,NaN,<NA>
4,2012-01,ANG MO KIO,2 ROOM,174,ANG MO KIO AVE 4,07 TO 09,45.0,Improved,1986,226000.0,ResaleFlatPricesBasedonApprovalDate2000Feb2012...,NaN,<NA>


## Next steps

The remaining stages of Part 1 (still pending):

* **§3 Profile.** Custom data profiling: per-column dtypes, null counts, value distributions, suspicious-pattern flags. Hand-rolled rather than `ydata-profiling` to keep the notebook focused on the rules we'll validate against.
* **§4 Validate.** Hand-rolled rule functions deriving allowed sets from the master's statistical properties (Date, Town, Flat Type, Flat Model, storey_range), plus numeric bounds and null-key checks. Failed rows go to `data/failed/validation_failures.csv`.
* **§5 Clean.** Dedupe on the composite key (highest `resale_price` wins), recompute `remaining_lease` as of today, flag anomalous prices via asymmetric IQR on price-per-sqm grouped by `(town, flat_type, year)`. Output: `data/cleaned/hdb_resale_cleaned.{csv,parquet}`.

**Out of scope (this pass):** Resale Identifier column, hashed identifier, `data/transformed/`, `data/hashed/`, Part 2 architecture diagrams.